# Liu2024 - TWFB+DGFMDM reproduction (the 72% method, standalone)

A faithful **approximation** of the Liu2024 winning pipeline: per-subject optimal **T**ime-**W**indow +
**F**ilter-**B**ank selection feeding a **D**iscriminant **G**eodesic **F**iltering + **M**inimum
**D**istance to Riemannian **M**ean classifier (DGFMDM ~= pyRiemann `FgMDM`).

Honest caveat: this is not a bit-exact reimplementation (Liu use a backtracking search + LTSA dimension
reduction; here the per-subject TW/FB selection is a grid search with inner CV, and `FgMDM` stands in for
DGFMDM). Exact reproduction of 72% is unlikely - your own CSP/FBCSP repro already came in ~5 points under
the paper. Treat this as a strong, comparable Riemannian baseline run under the *same* within-subject CV
as the hybrid notebook, so the two numbers are directly comparable.

Pipeline per subject: crop MI window -> for each (time sub-window x frequency band) compute spatial
covariances -> inner-CV select the best (window, band) with FgMDM -> refit on the train fold -> predict.

> Requires pyRiemann: `pip install pyriemann`

# 1. Setup

In [1]:
import os, re, sys, json, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from scipy.io import loadmat
import mne
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

try:
    from pyriemann.estimation import Covariances
    from pyriemann.classification import FgMDM
except Exception as e:
    raise ImportError("pyRiemann is required: pip install pyriemann") from e
print("ok | python", sys.version.split()[0])


ok | python 3.11.15


# 2. Configuration

In [2]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-twfb-dgfmdm"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "subjects_to_use": None, "exclude_subjects": [],

    # ---- signal / window (shared with the hybrid notebook for comparability) ----
    "source_unit": "microvolts", "reference_mode": "average",
    "resample_sfreq": 128, "mi_window_start_s": 2.0, "target_window_samples": 537,
    "demean_mode": "baseline_window_mean", "baseline_window_s": [0.0, 2.0],

    # ---- TWFB grid ----
    # Full paper grid: 7 time windows x 19 bands. quick_mode subsets it for fast iteration.
    "quick_mode": False,
    "time_windows_s": [[0.0,1.0],[0.5,1.5],[1.0,2.0],[1.5,2.5],[2.0,3.0],[2.5,3.5],[3.0,4.0]],
    "filter_bands": [[8,12],[9,13],[10,14],[11,15],[12,16],[13,17],[14,18],[15,19],[16,20],
                     [17,21],[18,22],[19,23],[20,24],[21,25],[22,26],[23,27],[24,28],[25,29],[26,30]],
    "quick_time_windows_s": [[0.0,2.0],[1.0,3.0],[2.0,4.0]],
    "quick_filter_bands": [[8,12],[12,16],[16,24],[24,30]],

    # ---- evaluation ----
    "cv_splits": 5, "inner_cv_splits": 3, "cov_estimator": "oas",
    "seed": 2026,
}
LIU_SFREQ=500; SFREQ=float(CONFIG["resample_sfreq"]); WIN=int(CONFIG["target_window_samples"])
TIME_WINDOWS = CONFIG["quick_time_windows_s"] if CONFIG["quick_mode"] else CONFIG["time_windows_s"]
BANDS        = CONFIG["quick_filter_bands"]   if CONFIG["quick_mode"] else CONFIG["filter_bands"]
ART=Path(CONFIG["artifact_dir"])/datetime.now().strftime("%Y%m%d_%H%M"); ART.mkdir(parents=True,exist_ok=True)
with open(ART/"config.json","w") as f: json.dump(CONFIG,f,indent=2,default=str)
print(f"grid: {len(TIME_WINDOWS)} windows x {len(BANDS)} bands = {len(TIME_WINDOWS)*len(BANDS)} combos | "
      f"quick_mode={CONFIG['quick_mode']}")


grid: 7 windows x 19 bands = 133 combos | quick_mode=False


# 3. Data: loader + preprocessing

In [3]:
SOURCE_EEG_INDICES=[i for i in range(30) if i!=17]
NAMES30=["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4",
         "CPz","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
EEG_NAMES=[n for i,n in enumerate(NAMES30) if i!=17]

def find_mats(root): root=Path(root); return sorted(root.rglob("*.mat")) if root.exists() else []
def sid_from_path(p):
    m=re.search(r"sub[-_ ]?(\d{1,2})",str(p),re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+",Path(p).stem)[-1])
def _walk(o,pre=""):
    if isinstance(o,dict):
        for k,v in o.items():
            if str(k).startswith("__"): continue
            n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif hasattr(o,"_fieldnames"):
        for k in o._fieldnames:
            v=getattr(o,k); n=f"{pre}.{k}" if pre else str(k); yield n,v; yield from _walk(v,n)
    elif isinstance(o,np.ndarray):
        if o.dtype==object and o.size==1: yield from _walk(o.item(),pre)
        elif o.dtype==object:
            for idx,it in np.ndenumerate(o): yield from _walk(it,f"{pre}{idx}")
def load_subject(path):
    mat=loadmat(path,squeeze_me=True,struct_as_record=False)
    arrs=[(n,np.asarray(v)) for n,v in _walk(mat) if isinstance(v,np.ndarray) and v.dtype!=object]
    raws=[a for n,a in arrs if a.ndim==3]
    labs=[a for n,a in arrs if a.ndim in (1,2) and np.asarray(a).size in (39,40)]
    if not raws or not labs: raise KeyError(path)
    raw=max(raws,key=lambda a:max(a.shape)); labels=np.asarray(labs[0]).astype(int).ravel()
    # normalise to trials x ch x time
    if raw.shape[0] not in (39,40):
        ax=[i for i,s in enumerate(raw.shape) if s in (39,40)]
        if ax: raw=np.moveaxis(raw,ax[0],0)
    if int(np.argmax(raw.shape[1:])+1)!=2: raw=np.moveaxis(raw,int(np.argmax(raw.shape[1:])+1),2)
    return raw.astype(np.float64), labels
def to_zero(labels):
    u=set(np.unique(labels).tolist())
    if u.issubset({1,2}): return labels-1
    if u.issubset({0,1}): return labels
    raise ValueError(u)
def info128(): 
    info=mne.create_info(EEG_NAMES,SFREQ,["eeg"]*len(EEG_NAMES)); return info
def preprocess(raw, labels):
    X=raw[:,SOURCE_EEG_INDICES,:].astype(np.float64); n=X.shape[0]
    b0,b1=CONFIG["baseline_window_s"]; s0,s1=int(b0*LIU_SFREQ),int(b1*LIU_SFREQ)
    if CONFIG["demean_mode"]=="baseline_window_mean": X=X-X[:,:,s0:s1].mean(-1,keepdims=True)
    cont=(X*1e-6).transpose(1,0,2).reshape(len(SOURCE_EEG_INDICES),-1)
    r=mne.io.RawArray(cont,mne.create_info(EEG_NAMES,LIU_SFREQ,["eeg"]*len(EEG_NAMES)),verbose=False)
    if CONFIG["reference_mode"]=="average": r.set_eeg_reference("average",projection=False,verbose=False)
    r.resample(SFREQ,verbose=False)
    d=r.get_data()*1e6; per=d.shape[1]//n; d=d[:,:n*per]
    Xrs=d.reshape(len(SOURCE_EEG_INDICES),n,per).transpose(1,0,2)
    st=int(round(CONFIG["mi_window_start_s"]*SFREQ)); sp=st+WIN
    return Xrs[:,:,st:sp].astype(np.float64), to_zero(labels).astype(int)

MATS=find_mats(CONFIG["source_extract_dir"])
if not MATS: raise FileNotFoundError(CONFIG["source_extract_dir"])
use=None if CONFIG["subjects_to_use"] is None else set(CONFIG["subjects_to_use"]); excl=set(CONFIG["exclude_subjects"])
SUBJECTS={}
for p in MATS:
    s=sid_from_path(p)
    if (use is not None and s not in use) or s in excl: continue
    raw,lab=load_subject(p); SUBJECTS[s]=preprocess(raw,lab)
ALL=sorted(SUBJECTS); print(f"loaded {len(ALL)} subjects | X={SUBJECTS[ALL[0]][0].shape}")


loaded 50 subjects | X=(40, 29, 537)


# 4. TWFB+DGFMDM per subject

For each subject and outer CV fold: precompute band-filtered signals, then for every (time-window, band)
combination compute covariances and score `FgMDM` with an inner CV on the training fold. Select the best
combination (the per-subject TW+FB choice), refit on the full training fold, and predict the test fold.

In [4]:
def bandpass(X, lo, hi):
    # X: trials x C x T  -> filtered copy
    return mne.filter.filter_data(X, SFREQ, lo, hi, method="fir", phase="zero",
                                  fir_design="firwin", verbose=False)

def win_samples(w):
    a=int(round(w[0]*SFREQ)); b=int(round(w[1]*SFREQ)); b=min(b,WIN); return a,b

def covs(Xseg):
    return Covariances(estimator=CONFIG["cov_estimator"]).transform(Xseg)

def inner_score(cov, y, splits, seed):
    skf=StratifiedKFold(n_splits=splits, shuffle=True, random_state=seed)
    accs=[]
    for tr,va in skf.split(cov, y):
        try:
            clf=FgMDM().fit(cov[tr], y[tr])
            accs.append(balanced_accuracy_score(y[va], clf.predict(cov[va])))
        except Exception:
            accs.append(0.5)
    return float(np.mean(accs))

def run_subject(sid):
    X, y = SUBJECTS[sid]
    outer=StratifiedKFold(n_splits=CONFIG["cv_splits"], shuffle=True, random_state=CONFIG["seed"])
    y_true_all, y_pred_all = [], []
    for fold,(tr,te) in enumerate(outer.split(X, y)):
        # precompute band-filtered signals once per fold
        band_filt = {bi: bandpass(X, lo, hi) for bi,(lo,hi) in enumerate(BANDS)}
        best, best_score = None, -1.0
        for bi,(lo,hi) in enumerate(BANDS):
            Xb = band_filt[bi]
            for wi,w in enumerate(TIME_WINDOWS):
                a,b = win_samples(w)
                cov = covs(Xb[:, :, a:b])
                sc = inner_score(cov[tr], y[tr], CONFIG["inner_cv_splits"], CONFIG["seed"]+fold)
                if sc > best_score: best_score, best = sc, (bi, wi)
        bi, wi = best
        a,b = win_samples(TIME_WINDOWS[wi])
        cov = covs(band_filt[bi][:, :, a:b])
        clf = FgMDM().fit(cov[tr], y[tr])
        y_true_all.extend(y[te].tolist()); y_pred_all.extend(clf.predict(cov[te]).tolist())
    yt,yp=np.array(y_true_all),np.array(y_pred_all)
    return {"subject":int(sid),"balanced_accuracy":float(balanced_accuracy_score(yt,yp)),
            "accuracy":float(accuracy_score(yt,yp)),
            "confusion":confusion_matrix(yt,yp,labels=[0,1]).tolist(),
            "y_true":yt.tolist(),"y_pred":yp.tolist()}


# 5. Run + aggregate

In [5]:
import time
RESULTS=[]
t0=time.time()
for i,s in enumerate(ALL,1):
    r=run_subject(s); RESULTS.append(r)
    print(f"[{i:2d}/{len(ALL)}] subj {s:2d}  BA={r['balanced_accuracy']*100:5.1f}%  "
          f"acc={r['accuracy']*100:5.1f}%  ({(time.time()-t0)/60:.1f} min elapsed)")
with open(ART/"twfb_dgfmdm_results.json","w") as f: json.dump(RESULTS,f,indent=2)

ba=np.array([r["balanced_accuracy"] for r in RESULTS])
yt=np.concatenate([r["y_true"] for r in RESULTS]); yp=np.concatenate([r["y_pred"] for r in RESULTS])
print("="*60)
print(f"TWFB+DGFMDM (FgMDM) | quick_mode={CONFIG['quick_mode']} | {len(ALL)} subjects, {CONFIG['cv_splits']}-fold within-subject")
print(f"  mean per-subject balanced accuracy: {ba.mean()*100:.2f}%  (SD {ba.std()*100:.2f})")
print(f"  global pooled balanced accuracy:    {balanced_accuracy_score(yt,yp)*100:.2f}%")
print(f"  Liu paper TWFB+DGFMDM target:       72.21%")
print("="*60)
df=pd.DataFrame([{"subject":r["subject"],"BA_%":round(r["balanced_accuracy"]*100,1),
                  "acc_%":round(r["accuracy"]*100,1)} for r in RESULTS]).sort_values("subject")
df.to_csv(ART/"twfb_dgfmdm_per_subject.csv",index=False); df.reset_index(drop=True)


[ 1/50] subj  1  BA= 52.5%  acc= 52.5%  (1.7 min elapsed)
[ 2/50] subj  2  BA= 47.5%  acc= 47.5%  (3.4 min elapsed)
[ 3/50] subj  3  BA= 55.0%  acc= 55.0%  (5.1 min elapsed)
[ 4/50] subj  4  BA= 50.0%  acc= 50.0%  (6.9 min elapsed)
[ 5/50] subj  5  BA= 62.5%  acc= 62.5%  (8.6 min elapsed)
[ 6/50] subj  6  BA= 47.5%  acc= 47.5%  (10.4 min elapsed)
[ 7/50] subj  7  BA= 75.0%  acc= 75.0%  (12.2 min elapsed)
[ 8/50] subj  8  BA= 32.5%  acc= 32.5%  (13.9 min elapsed)
[ 9/50] subj  9  BA= 45.0%  acc= 45.0%  (15.6 min elapsed)
[10/50] subj 10  BA= 50.0%  acc= 50.0%  (17.4 min elapsed)
[11/50] subj 11  BA= 47.5%  acc= 47.5%  (19.1 min elapsed)
[12/50] subj 12  BA= 50.0%  acc= 50.0%  (20.9 min elapsed)
[13/50] subj 13  BA= 70.0%  acc= 70.0%  (22.6 min elapsed)
[14/50] subj 14  BA= 65.0%  acc= 65.0%  (24.3 min elapsed)
[15/50] subj 15  BA= 57.5%  acc= 57.5%  (26.0 min elapsed)
[16/50] subj 16  BA= 35.0%  acc= 35.0%  (27.8 min elapsed)
[17/50] subj 17  BA= 50.0%  acc= 50.0%  (29.5 min elapsed)
[1

,subject,BA_%,acc_%
0,1,52.5,52.5
1,2,47.5,47.5
2,3,55.0,55.0
3,4,50.0,50.0
4,5,62.5,62.5
5,6,47.5,47.5
6,7,75.0,75.0
7,8,32.5,32.5
8,9,45.0,45.0
9,10,50.0,50.0


## Notes
- Set `quick_mode=False` (default) for the full 7x19 grid - faithful but slow (~minutes/subject). `quick_mode=True`
  is for fast iteration and will score lower.
- The protocol here (5-fold within-subject, balanced accuracy) matches the hybrid notebook so the numbers
  are directly comparable. Liu used 60/40 x10 and reported plain accuracy, so a few points of difference
  vs 72.21% is expected even before reproduction gaps.